# Multi-Head Attention Explorer

## How Multi-Head Attention Works

In a transformer model, each layer contains multiple **attention heads** that operate in parallel. Each head learns to attend to different aspects of the input:

- **Syntactic heads** learn grammatical structure — a head might consistently attend from a verb to its subject, or from an adjective to the noun it modifies.
- **Semantic heads** capture meaning relationships — linking words that are semantically related even when far apart in the sentence.
- **Positional/temporal heads** track word order and sequential relationships, such as connecting temporal markers ("last night") to the events they describe.
- **Coreference heads** link pronouns back to their antecedents, or connect different mentions of the same entity.

An attention map is a matrix where entry `(i, j)` represents how much token `i` "attends to" token `j`. Each row sums to 1 (it's a probability distribution via softmax). By examining these maps across different heads and layers, we can see what linguistic patterns the model has learned.

**BERT** has 12 layers with 12 attention heads each, giving 144 distinct attention patterns to explore. Lower layers tend to capture local syntax; higher layers capture more abstract, long-range relationships.

This notebook lets you interactively explore these attention patterns.

In [ ]:
# Dependencies
import torch
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from transformers import BertTokenizer, BertModel

%matplotlib inline
plt.rcParams['figure.dpi'] = 100
print("All imports loaded successfully.")

In [ ]:
# Load BERT with attention outputs enabled
MODEL_NAME = "bert-base-uncased"

tokenizer = BertTokenizer.from_pretrained(MODEL_NAME)
model = BertModel.from_pretrained(MODEL_NAME, output_attentions=True)
model.eval()

print(f"Model: {MODEL_NAME}")
print(f"Layers: {model.config.num_hidden_layers}")
print(f"Attention heads per layer: {model.config.num_attention_heads}")
print(f"Total attention heads: {model.config.num_hidden_layers * model.config.num_attention_heads}")

In [ ]:
# ============================================================
# INTERACTIVE INPUT — Change this text and re-run the notebook
# ============================================================
INPUT_TEXT = "Nacho slept on the couch last night."
# ============================================================

print(f"Input text: \"{INPUT_TEXT}\"")

In [ ]:
# Tokenize and extract attention maps
inputs = tokenizer(INPUT_TEXT, return_tensors="pt")

with torch.no_grad():
    outputs = model(**inputs)

# outputs.attentions is a tuple of tensors, one per layer
# Each tensor has shape (batch_size, num_heads, seq_len, seq_len)
attention_maps = outputs.attentions
token_ids = inputs["input_ids"][0]
token_labels = [tokenizer.decode([tid]) for tid in token_ids]

num_layers = len(attention_maps)
num_heads = attention_maps[0].shape[1]
seq_len = attention_maps[0].shape[2]

print(f"Tokens ({seq_len}): {token_labels}")
print(f"Attention tensor shape per layer: {attention_maps[0].shape}")
print(f"Available: {num_layers} layers x {num_heads} heads")

---
## Visualization 1: Single Attention Head Heatmap

Select a specific layer and head to inspect. The heatmap shows how much each token (row) attends to every other token (column). Bright cells indicate strong attention.

In [ ]:
# ============================================================
# SELECT LAYER AND HEAD — Change these to explore
# Layer range: 0 to 11, Head range: 0 to 11
# ============================================================
LAYER = 0
HEAD = 0
# ============================================================

attention = attention_maps[LAYER][0, HEAD].numpy()

fig, ax = plt.subplots(figsize=(8, 8))
im = ax.imshow(attention, cmap="viridis", vmin=0, vmax=attention.max())

ax.set_xticks(np.arange(seq_len))
ax.set_yticks(np.arange(seq_len))
ax.set_xticklabels(token_labels, fontsize=11)
ax.set_yticklabels(token_labels, fontsize=11)
plt.setp(ax.get_xticklabels(), rotation=45, ha="right", rotation_mode="anchor")

ax.set_xlabel("Attended To", fontsize=12)
ax.set_ylabel("Attending From", fontsize=12)
ax.set_title(f"Attention Heatmap — Layer {LAYER}, Head {HEAD}", fontsize=14)

cbar = fig.colorbar(im, ax=ax, shrink=0.8)
cbar.set_label("Attention Weight", fontsize=11)

plt.tight_layout()
plt.show()

---
## Visualization 2: Comparing 4 Heads in One Layer

This 2x2 grid shows the first four heads from a single layer side by side. Notice how each head learns to focus on entirely different patterns — one might attend to the next word, another to punctuation, another to the subject, etc.

In [ ]:
# ============================================================
# SELECT LAYER AND WHICH 4 HEADS TO COMPARE
# ============================================================
GRID_LAYER = 0
GRID_HEADS = [0, 1, 2, 3]  # pick any 4 heads (0-11)
# ============================================================

fig, axes = plt.subplots(2, 2, figsize=(14, 14))
axes = axes.ravel()

for idx, head_idx in enumerate(GRID_HEADS):
    ax = axes[idx]
    attn = attention_maps[GRID_LAYER][0, head_idx].numpy()
    im = ax.imshow(attn, cmap="viridis", vmin=0, vmax=attn.max())

    ax.set_xticks(np.arange(seq_len))
    ax.set_yticks(np.arange(seq_len))
    ax.set_xticklabels(token_labels, fontsize=9)
    ax.set_yticklabels(token_labels, fontsize=9)
    plt.setp(ax.get_xticklabels(), rotation=45, ha="right", rotation_mode="anchor")

    ax.set_title(f"Layer {GRID_LAYER}, Head {head_idx}", fontsize=12)
    fig.colorbar(im, ax=ax, shrink=0.75)

fig.suptitle(f"4 Attention Heads from Layer {GRID_LAYER} — Same Input, Different Patterns",
             fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

---
## Visualization 3: Attention Rollout (Average Across Heads)

By averaging attention weights across all heads in a layer, we get the **overall attention pattern** for that layer. This smooths out individual head specializations and shows the aggregate information flow.

In [ ]:
# ============================================================
# SELECT LAYER FOR ATTENTION ROLLOUT
# ============================================================
ROLLOUT_LAYER = 0
# ============================================================

# Average attention across all heads for the selected layer
avg_attention = attention_maps[ROLLOUT_LAYER][0].mean(dim=0).numpy()

fig, ax = plt.subplots(figsize=(8, 8))
im = ax.imshow(avg_attention, cmap="viridis", vmin=0, vmax=avg_attention.max())

ax.set_xticks(np.arange(seq_len))
ax.set_yticks(np.arange(seq_len))
ax.set_xticklabels(token_labels, fontsize=11)
ax.set_yticklabels(token_labels, fontsize=11)
plt.setp(ax.get_xticklabels(), rotation=45, ha="right", rotation_mode="anchor")

ax.set_xlabel("Attended To", fontsize=12)
ax.set_ylabel("Attending From", fontsize=12)
ax.set_title(f"Attention Rollout — Layer {ROLLOUT_LAYER} (mean of {num_heads} heads)",
             fontsize=14)

cbar = fig.colorbar(im, ax=ax, shrink=0.8)
cbar.set_label("Average Attention Weight", fontsize=11)

plt.tight_layout()
plt.show()

---
## Visualization 4: Attention Entropy per Head

**Entropy** measures how spread out (diffuse) or concentrated (sharp) an attention distribution is.

- **Low entropy** = the head focuses strongly on just one or a few tokens. These heads have likely learned a specific pattern (e.g., "always attend to the previous word").
- **High entropy** = attention is spread roughly evenly across all tokens. These heads capture broad, global context.

We compute entropy for each head (averaged over all query positions) in a given layer.

In [ ]:
# ============================================================
# SELECT LAYER FOR ENTROPY ANALYSIS
# ============================================================
ENTROPY_LAYER = 0
# ============================================================

# Compute entropy for each head in the selected layer
# Entropy H = -sum(p * log(p)) for each row, then average over rows
layer_attention = attention_maps[ENTROPY_LAYER][0].numpy()  # shape: (num_heads, seq_len, seq_len)

entropies = []
for h in range(num_heads):
    head_attn = layer_attention[h]  # shape: (seq_len, seq_len)
    # Clip to avoid log(0)
    head_attn_clipped = np.clip(head_attn, 1e-12, 1.0)
    # Entropy per query position (per row)
    row_entropies = -np.sum(head_attn_clipped * np.log2(head_attn_clipped), axis=1)
    # Average entropy across all query positions
    entropies.append(np.mean(row_entropies))

entropies = np.array(entropies)

# Maximum possible entropy for this sequence length (uniform distribution)
max_entropy = np.log2(seq_len)

# Color bars by relative entropy: low = focused (cool), high = diffuse (warm)
norm = mcolors.Normalize(vmin=entropies.min(), vmax=entropies.max())
colors = plt.cm.RdYlBu_r(norm(entropies))

fig, ax = plt.subplots(figsize=(12, 5))
bars = ax.bar(range(num_heads), entropies, color=colors, edgecolor="black", linewidth=0.5)

ax.axhline(y=max_entropy, color="gray", linestyle="--", linewidth=1,
           label=f"Max entropy (uniform) = {max_entropy:.2f} bits")

ax.set_xlabel("Head Index", fontsize=12)
ax.set_ylabel("Average Entropy (bits)", fontsize=12)
ax.set_title(f"Attention Entropy per Head — Layer {ENTROPY_LAYER}", fontsize=14)
ax.set_xticks(range(num_heads))
ax.set_xticklabels([f"H{i}" for i in range(num_heads)], fontsize=10)
ax.legend(fontsize=10)
ax.set_ylim(0, max_entropy * 1.15)

# Annotate the sharpest and most diffuse heads
sharpest = np.argmin(entropies)
most_diffuse = np.argmax(entropies)
ax.annotate(f"Sharpest\n({entropies[sharpest]:.2f} bits)",
            xy=(sharpest, entropies[sharpest]),
            xytext=(sharpest, entropies[sharpest] + max_entropy * 0.08),
            ha="center", fontsize=9, fontweight="bold",
            arrowprops=dict(arrowstyle="->", color="black", lw=1))
ax.annotate(f"Most diffuse\n({entropies[most_diffuse]:.2f} bits)",
            xy=(most_diffuse, entropies[most_diffuse]),
            xytext=(most_diffuse, entropies[most_diffuse] + max_entropy * 0.08),
            ha="center", fontsize=9, fontweight="bold",
            arrowprops=dict(arrowstyle="->", color="black", lw=1))

plt.tight_layout()
plt.show()

print(f"\nEntropy summary for Layer {ENTROPY_LAYER}:")
print(f"  Min entropy: Head {sharpest} = {entropies[sharpest]:.3f} bits (most focused)")
print(f"  Max entropy: Head {most_diffuse} = {entropies[most_diffuse]:.3f} bits (most diffuse)")
print(f"  Max possible entropy: {max_entropy:.3f} bits (uniform over {seq_len} tokens)")

---
## What to Look For

### Patterns in individual heads
- **Diagonal patterns**: The head attends to the current token or its immediate neighbors (positional attention).
- **Vertical stripes**: All tokens attend strongly to a single token, often `[CLS]` or `[SEP]` (these are "sink" tokens that aggregate global information).
- **Block patterns**: Contiguous spans attend to each other, possibly capturing phrase structure.
- **Off-diagonal connections**: Long-range links between specific token pairs may indicate learned syntactic or semantic dependencies.

### Across layers
- **Layer 0-3** (early): Tend to show local, positional patterns and simple syntactic relationships.
- **Layer 4-8** (middle): More complex syntactic patterns emerge — subject-verb, modifier-noun links.
- **Layer 9-11** (late): Abstract, task-specific patterns; attention often concentrates on `[CLS]`.

### Entropy insights
- Heads with very low entropy in early layers often implement "attend to previous token" or "attend to self."
- Heads with moderate entropy in middle layers often capture the most interpretable linguistic patterns.
- High-entropy heads spread attention broadly and may function as residual or normalization mechanisms.

### Experiments to try
1. **Change the input text** to a longer sentence with relative clauses (e.g., "The cat that the dog chased ran away.") and see if any head learns the relative clause structure.
2. **Compare layers**: Set `LAYER=0`, then `LAYER=6`, then `LAYER=11` for the same head — watch patterns evolve from local to global.
3. **Pronoun resolution**: Try "Alice told Bob that she would help him." and look for heads linking "she" to "Alice" and "him" to "Bob."
4. **Negation**: Compare "The movie was good" vs. "The movie was not good" — see which heads change their attention patterns around "not."
5. **Low-entropy heads**: Use the entropy chart to identify the sharpest head, then visualize it — what pattern has it learned?